# Agentic AI Exercise 1

In [1]:
%pip install -q langchain langchain_openai langchain_community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.9/515.9 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [2]:
import os

In [60]:
try:
    # In Colab? read from userdata (secrets)
    from google.colab import userdata
    ON_COLAB = True
    os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
except ImportError:
    # Load `.env` file (locally)
    from dotenv import load_dotenv
    load_dotenv(override=True)

## **Task 1**

# Importing ChatOpenAI and defining the model

In [6]:
from langchain_openai import ChatOpenAI

In [64]:
model_nvidia_nano_lowtemp= ChatOpenAI(
    model = 'nvidia/nemotron-3-nano-30b-a3b:free',
    temperature=0,
    base_url= 'https://openrouter.ai/api/v1',
    api_key=os.environ.get("OPENROUTER_API_KEY"),

)

model_nvidia_nano_hightemp= ChatOpenAI(
    model = 'nvidia/nemotron-3-nano-30b-a3b:free',
    temperature=0.8,
    base_url= 'https://openrouter.ai/api/v1',
    api_key=os.environ.get("OPENROUTER_API_KEY"),

)

In [15]:
message_lowtemp = model_nvidia_nano_lowtemp.invoke("Write a one sentence tagline for a coffee shop")
message_highemp = model_nvidia_nano_hightemp.invoke("Write a one sentence tagline for a coffee shop")

In [16]:
print(message_lowtemp.content)

"Sip themoment, savor the magic—your daily brew, elevated."


In [17]:
print(message_highemp.content)

"Sip the moment, savor the extraordinary—in every cup, we brew your perfect pause."


In [21]:
message_lowtemp = model_nvidia_nano_lowtemp.invoke("Describe the ocean using only emojis")
message_highemp = model_nvidia_nano_hightemp.invoke("Describe the ocean using only emojis")

In [22]:
print(message_lowtemp.content)

🌊🌊🌊🌅🐚🐠🐙🐬🦈🌞🏖️🏝️🐬🐳🐟🦀🦐🪸🪸🪸


In [23]:
print(message_highemp.content)

🌊🌊🌊🌊🌊🌊🌊  🐠🐡🐟🦐🦞🐙🦑  🐋🐳🦈🦞  
🐚🐚🐚  
🌅🌊☀️  
🌅🌊🌅  
🌊🌊🌊🌊🌊🌊🌊  
🌊🌊🌊🌊🌊🌊🌊  
🌊🌊🌊🌊🌊🌊🌊


In [24]:
message_lowtemp = model_nvidia_nano_lowtemp.invoke("Describe the ocean using only emojis, format it is JSON")
message_highemp = model_nvidia_nano_hightemp.invoke("Describe the ocean using only emojis, Format it as JSON")

In [25]:
print(message_lowtemp.content)

{
  "ocean": "🌊🐚🐠🐬🦈🌅🌊🌊🌊"
}


In [26]:
print(message_highemp.content)

{
  "ocean": "🌊🐚🦈🐬🐟🐳🌅✨🐙🌈"
}




---



## Task 2: Sentiment Analysis


In [65]:
from pydantic import BaseModel, Field
from langchain.messages import SystemMessage, HumanMessage

class Sentiment(BaseModel):
  sentiment: str = Field(..., description="Output ONLY one of: 'positive', 'neutral', 'negative', Only One Word")

In [66]:
sentiment_llm = model_nvidia_nano_lowtemp.with_structured_output(Sentiment)

In [67]:
#it was not working correctly and hullucinated so i had to add systemMessage to make it stricter

sentences = [
    "Kindness creates lasting joy.",
    "Success rewards persistent effort.",
    "Pessemestic all the time.",
    "The storm caused damage!",
    "The clock ticks steadily.",
]

for sentence in sentences:
    messages = [
        SystemMessage("You are a sentiment classifier. You must respond with ONLY one word: positive, neutral, or negative. Nothing else."),
        HumanMessage(sentence)
    ]
    result = sentiment_llm.invoke(messages)
    print(f"{sentence} → {result.sentiment}")

Kindness creates lasting joy. → positive
Success rewards persistent effort. → positive
Pessemestic all the time. → negative
The storm caused damage! → negative
The clock ticks steadily. → neutral


## Task 3: Categorization


In [75]:
from typing import List #got it from my VA for better structured output

class Tags(BaseModel):
    tags: List[str] = Field(..., description="List of relevant tags. Output ONLY from: 'cars', 'shopping', 'sports', 'study', 'work'.")

In [76]:
tags_llm= model_nvidia_nano_lowtemp.with_structured_output(Tags)

In [83]:
#it was not working correctly and hullucinated so i had to add systemMessage to make it stricter

sentences = [
    "That restoration looks incredible; you have a real talent for mechanics.",
    "I found the perfect gift today! The staff was incredibly helpful.",
    "Great game today! Your teamwork was the key to that victory.",
    "Learning together helped me finally grasp these concepts. Thank you!"
]


for sentence in sentences:
    messages = [
        SystemMessage("You are a Categorizor. You must respond with ONLY with categories givin."),
        HumanMessage(sentence)
    ]
    result = tags_llm.invoke(messages)
    print(f"{sentence} → {result.tags}")

That restoration looks incredible; you have a real talent for mechanics. → ['Compliment']
I found the perfect gift today! The staff was incredibly helpful. → ['service']
Great game today! Your teamwork was the key to that victory. → ['teamwork']
Learning together helped me finally grasp these concepts. Thank you! → ['Positive Feedback']


## Task4: Data Extraction

In [80]:

class CV(BaseModel):
    name: str = Field(None, description="Full name of the candidate.")
    email: str = Field(None, description="Email address of the candidate.")
    skills: str = Field(None, description="Skills of the candidate.")
    experience: str = Field(None, description="Work experience of the candidate.")

In [81]:
cv_llm= model_nvidia_nano_lowtemp.with_structured_output(CV)

In [ ]:
cv_llm = model_nvidia_nano_lowtemp.with_structured_output(CV)
cv_path="Data/CV.txt"
# Read the CV text file
with open(cv_path, "r") as f:
    cv_text = f.read()
messages = [
        SystemMessage("You are a cv data extractor extract info based on the CV given"),
        HumanMessage(sentence)
    ]
result = cv_llm.invoke(cv_text)

print("Name:", result.name)
print(i a"Email:", result.email)
print("Skills:", result.skills)
print("Experience:", result.experience)


Name: JohnSmith
Email: john.smith@email.com
Skills: Python, FastAPI, React, PostgreSQL, AWS, Git, Docker
Experience: Software Engineer at TechCorp (2021-2024)
- Developed and maintained RESTful APIs using Python and FastAPI, handling 10K+ daily requests
- Designed and optimized PostgreSQL schemas, improving query performance by 30%
- Managed cloud infrastructure on AWS (EC2, S3, RDS), reducing operational costs by 15%

Junior Developer at StartupXYZ (2019-2021)
- Built responsive UI components with React, contributing to a 20% increase in user engagement
- Collaborated with product, design, and backend teams in Agile sprints to deliver features on schedule
- Utilized Git for version control and Docker for containerized development environments

Education
- B.Sc. Computer Science, State University (2019)
- Relevant coursework: Data Structures, Web Development, Database Systems
